# Lesson 01 - Introdução aos Agentes de IA

Bem-vindo à primeira aula do curso **Agentes de IA para Iniciantes**!

Um **agente de IA** é um programa que utiliza um modelo de linguagem grande (LLM) como seu motor de raciocínio e pode tomar *ações* no mundo real — chamando APIs, consultando bancos de dados ou executando código — para alcançar um objetivo em nome de um usuário.

Neste notebook você irá construir seu primeiro agente: um **Agente de Viagens** que recomenda destinos de férias. Durante o processo, você aprenderá a:

1. Conectar-se ao Azure AI Foundry Agent Service usando o **Microsoft Agent Framework**.
2. Fornecer ao agente uma **ferramenta** — uma função Python simples que ele pode chamar.
3. Executar o agente e inspecionar sua resposta.
4. Transmitir a resposta do agente token por token.


## Configuração

Antes de executar este notebook, certifique-se de que você tem:

1. **Um projeto Azure AI Foundry** com um modelo de chat implantado (por exemplo, `gpt-4o-mini`).
2. **Conectado com o Azure CLI** — execute `az login` no seu terminal.
3. **Definido as variáveis de ambiente necessárias:**
   - `AZURE_AI_PROJECT_ENDPOINT` — o endpoint do seu projeto Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — o nome do seu modelo implantado.

A célula abaixo instala os pacotes Python de que você precisa.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Criando Seu Primeiro Agente

Um agente precisa de duas coisas:

- **Instruções** que lhe dizem *quem ele é* e *como se comportar* (um prompt do sistema).
- **Ferramentas** — funções Python decoradas com `@tool` que o agente pode chamar para obter informações ou realizar ações.

Abaixo definimos uma ferramenta simples que retorna uma lista de destinos de férias populares. O agente usará essa ferramenta quando um usuário solicitar recomendações de viagem.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Respostas em Streaming

Para uma experiência mais interativa, você pode **transmitir** a resposta do agente. Em vez de esperar pela resposta completa, o agente fornece pedaços de texto conforme são gerados. Isso é especialmente útil em interfaces de chat onde você deseja exibir a saída em tempo real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Resumo

Nesta lição você aprendeu a:

- **Criar um provedor** que conecta ao Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Definir uma ferramenta** usando o decorador `@tool` para que o agente possa chamar suas funções Python.
- **Executar o agente** com uma mensagem do usuário e imprimir sua resposta.
- **Transmitir respostas** para saída em tempo real.

Na próxima lição, exploraremos frameworks agentic com mais profundidade e aprenderemos a fornecer ferramentas mais poderosas e capacidades de raciocínio em múltiplas etapas para os agentes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte oficial. Para informações críticas, recomenda-se tradução profissional feita por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
